In [1]:
import pandas as pd
import numpy as np

In [2]:
u = np.array([2, 3, 4])

In [3]:
v = np.array([3, 4, 5])

In [4]:
u*v

array([ 6, 12, 20])

In [5]:
u.dot(v.transpose())

np.int64(38)

In [7]:
a = np.random.randint(2, 7, (2,2))
np.linalg.inv(a)

array([[ 0.5, -0.4],
       [-0.5,  0.6]])

In [7]:
x = np.array(
    [[4, 6, 7],
    [7, 9, 5],
    [2, 4, 5]]
)
y = np.array([3, 5, 6])
ones = np.ones(3)

np.column_stack([ones, x])

array([[1., 4., 6., 7.],
       [1., 7., 9., 5.],
       [1., 2., 4., 5.]])

In [69]:
def linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XT = X.T
    XTX = XT.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(XT.dot(y))
    return w[0], w[1:]

def linear_regression_reg(X, y, r = 0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XT = X.T
    XTX = XT.dot(X)
    XTX = XTX + r*np.eye(XTX.shape[0])
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(XT.dot(y))
    return w[0], w[1:]

def prepare_X(df_, column_name, fillna_value):
    df_num = df_.copy()
    df_num[column_name] = df_num[column_name].fillna(fillna_value)
    return df_num

def rmse(y, y_pred):
    se = (y - y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)

def split_data(df_, seed):
    n = len(df_)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df_.iloc[idx[:n_train]]
    df_val = df_.iloc[idx[n_train:n_train + n_val]]
    df_test = df_.iloc[idx[n_train+n_val:]]
    
    print(f"df_train shape: {df_train.shape}")
    print(f"df_val shape: {df_val.shape}")
    print(f"df_test shape: {df_test.shape}")

    return df_train, df_val, df_test
    

In [18]:
linear_regression_reg(x, y, 1)

(np.float64(0.7191011235954896),
 array([-0.63081862,  0.80738363,  0.23434992]))

In [22]:
np.random.seed(24)
idx = np.array([2, 3, 4, 6, 4])
np.random.shuffle(idx)
idx

array([4, 3, 2, 6, 4])

In [50]:
df = pd.read_csv("../data/data.csv")
df = df.select_dtypes(include = "number")
df.head()

,Year,Engine HP,Engine Cylinders,Number of Doors,highway MPG,city mpg,Popularity,MSRP
0,2011,335.0,6.0,2.0,26,19,3916,46135
1,2011,300.0,6.0,2.0,28,19,3916,40650
2,2011,300.0,6.0,2.0,28,20,3916,36350
3,2011,230.0,6.0,2.0,28,18,3916,29450
4,2011,230.0,6.0,2.0,28,18,3916,34500


In [51]:
df.isna().sum()

Year                 0
Engine HP           69
Engine Cylinders    30
Number of Doors      6
highway MPG          0
city mpg             0
Popularity           0
MSRP                 0
dtype: int64

In [38]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(24)
idx = np.arange(n)
np.random.shuffle(idx)

In [48]:
print(n_train)
print(n_test)
print(n_val)

7150
2382
2382


In [49]:
idx

array([ 166, 8714, 8456, ..., 3473, 8535,  899], shape=(11914,))

In [ ]:
print(idx[:n_train])
print(idx[n_train:n_train + n_test])
print(idx[n_train+n_test:])

In [45]:
idx[:7150]

array([ 166, 8714, 8456, ..., 1683, 6351, 2559], shape=(7150,))

In [39]:
idx[:n_train]

array([ 166, 8714, 8456, ..., 1683, 6351, 2559], shape=(7150,))

In [46]:
df.iloc[[1, 4, 6]]

,Year,Engine HP,Engine Cylinders,Number of Doors,highway MPG,city mpg,Popularity,MSRP
1,2011,300.0,6.0,2.0,28,19,3916,40650
4,2011,230.0,6.0,2.0,28,18,3916,34500
6,2012,300.0,6.0,2.0,26,17,3916,44100


In [ ]:
df_= df.iloc[idx[:n_train]]


In [47]:
idx[n_train:n_train + n_val]

array([ 5381, 10239,  1693, ...,  5729,  4328,  3427], shape=(2382,))

In [ ]:
df_v = df.iloc[idx[n_train:n_train + n_val]]


In [ ]:
df_test = df.iloc[idx[n_train + n_val:]]

In [63]:
df.columns = df.columns.str.lower().str.replace(" ", "_").to_list()
df.columns

Index(['year', 'engine_hp', 'engine_cylinders', 'number_of_doors',
       'highway_mpg', 'city_mpg', 'popularity', 'msrp'],
      dtype='object')

In [ ]:
df_train, df_val, df_test = split_data(df, seed = 24)
y_train = df_train["msrp"].values
y_val = df_val["msrp"].values
y_test = df_test["msrp"].values

#option A, filling all missing values with 0
# number_of_doors
# engine_cylinders
# engine_hp

X_train_0 = prepare_X(df_train, "number_of_doors", 0)
X_train_0 = prepare_X(X_train_0, "engine_cylinders", 0)
X_train_0 = prepare_X(X_train_0, "engine_hp", 0)
print("X_tain head")
print(X_train_0.head())
print(X_train_0.isna().sum())

X_val_0 = prepare_X(df_val, "number_of_doors", 0)
X_val_0 = prepare_X(X_val_0, "engine_cylinders", 0)
X_val_0 = prepare_X(X_val_0, "engine_hp", 0)
print("X_val head")
print(X_val_0.head())
print(X_val_0.isna().sum())



df_train shape: (7150, 8)
df_val shape: (2382, 8)
df_test shape: (2382, 8)
X_tain head
      year  engine_hp  engine_cylinders  number_of_doors  highway_mpg  \
166   1991      177.0               6.0              4.0           20   
8714  2011      212.0               0.0              4.0           23   
8456  2017      259.0               4.0              4.0           29   
6487  2016      386.0               8.0              4.0           24   
9819  2015      191.0               4.0              4.0           25   

      city_mpg  popularity   msrp  
166         15         617   2000  
8714        16         586  26795  
8456        21         155  31615  
6487        16         454  72520  
9819        19        1720  26100  
year                0
engine_hp           0
engine_cylinders    0
number_of_doors     0
highway_mpg         0
city_mpg            0
popularity          0
msrp                0
dtype: int64
X_val head
       year  engine_hp  engine_cylinders  number_of_doors 

## **linear regression without regularization**

In [85]:
w0, w = linear_regression(X_train_0, y_train)
y_pred = w0 + X_val_0.dot(w)
rmse_0 = round(rmse(y_pred, y_val), 7)
print("RMSE (fill with 0):", rmse_0)


RMSE (fill with 0): 7e-07


In [81]:
#option A, filling all missing values with 0
# number_of_doors
# engine_cylinders
# engine_hp

X_train_mean = prepare_X(df_train, "number_of_doors", df_train["number_of_doors"].mean())
X_train_mean = prepare_X(X_train_mean, "engine_cylinders", df_train["engine_cylinders"].mean())
X_train_mean = prepare_X(X_train_mean, "engine_hp", df_train["engine_hp"].mean())
print("X_tain_mean head")
print(X_train_mean.head())
print(X_train_mean.isna().sum())

X_val_mean = prepare_X(df_val, "number_of_doors", df_train["number_of_doors"].mean())
X_val_mean = prepare_X(X_val_mean, "engine_cylinders", df_train["engine_cylinders"].mean())
X_val_mean = prepare_X(X_val_mean, "engine_hp", df_train["engine_hp"].mean())
print("X_tain_mean head")
print(X_train_mean.head())
print(X_train_mean.isna().sum())


X_tain_mean head
      year  engine_hp  engine_cylinders  number_of_doors  highway_mpg  \
166   1991      177.0          6.000000              4.0           20   
8714  2011      212.0          5.635535              4.0           23   
8456  2017      259.0          4.000000              4.0           29   
6487  2016      386.0          8.000000              4.0           24   
9819  2015      191.0          4.000000              4.0           25   

      city_mpg  popularity   msrp  
166         15         617   2000  
8714        16         586  26795  
8456        21         155  31615  
6487        16         454  72520  
9819        19        1720  26100  
year                0
engine_hp           0
engine_cylinders    0
number_of_doors     0
highway_mpg         0
city_mpg            0
popularity          0
msrp                0
dtype: int64
X_tain_mean head
      year  engine_hp  engine_cylinders  number_of_doors  highway_mpg  \
166   1991      177.0          6.000000          

## **linear regression with regularization**

In [84]:
w0, w = linear_regression_reg(X_train_0, y_train, 0.1)
y_pred = w0 + X_val_0.dot(w)
rmse_mean = round(rmse(y_pred, y_val), 7)
print("RMSE (fill with 0):", rmse_mean)

RMSE (fill with 0): 3e-07


In [86]:
rmse_0 > rmse_mean

np.True_